# New Section

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import fsolve
from scipy.linalg import eigvals

# ============================================================
# Model parameters (from Table 1, reference 70 kg adult)
# ============================================================
params = {
    'kabs': 1.0,          # h^-1
    'Q_GU': 1.5,          # L/h
    'Q_KI': 1.0,
    'Q_HA': 0.25,
    'Q_SP': 0.3,
    'Q_LI': 2.05,
    'Q_LU': 6.0,
    'Q_PER': 3.0,
    'V_GU': 0.5,          # L
    'V_KI': 0.3,
    'V_LI': 1.5,
    'V_AR': 2.0,
    'V_VE': 4.0,
    'BP': 0.8,
    'Kp_GU': 3.0,
    'Kp_KI': 4.0,
    'Kp_LI': 5.0,
    'Vmax_KI': 0.5,       # L/h
    'Km_KI': 0.1,         # mg/L
    'fu': 0.2,
    'Vmax_LI': 55.0,      # will vary
    'Km_LI': 0.5,         # mg/L (default)
}

# Effective volumes
def Veff(V, BP, Kp):
    return V / (BP * Kp)

params['Veff_GU'] = Veff(params['V_GU'], params['BP'], params['Kp_GU'])
params['Veff_KI'] = Veff(params['V_KI'], params['BP'], params['Kp_KI'])
params['Veff_LI'] = Veff(params['V_LI'], params['BP'], params['Kp_LI'])

# ============================================================
# ODE system - returns numpy array
# ============================================================
def rhs(t, y, p):
    A_GL, A_GU, A_KI, A_LI, A_AR, A_VE = y
    C_AR = A_AR / p['V_AR']
    C_VE = A_VE / p['V_VE']
    C_GU = A_GU / p['Veff_GU']
    C_KI = A_KI / p['Veff_KI']
    C_LI = A_LI / p['Veff_LI']

    dGL = -p['kabs'] * A_GL
    dGU = p['kabs'] * A_GL + p['Q_GU'] * (C_AR - C_GU)
    dKI = p['Q_KI'] * (C_AR - C_KI) - (p['Vmax_KI'] * p['fu'] * C_KI) / (p['Km_KI'] + p['fu'] * C_KI)
    dLI = p['Q_HA'] * C_AR + (p['Q_GU'] + p['Q_SP']) * C_GU - p['Q_LI'] * C_LI \
          - (p['Vmax_LI'] * p['fu'] * C_LI) / (p['Km_LI'] + p['fu'] * C_LI)
    dAR = p['Q_LU'] * (C_VE - C_AR)
    dVE = p['Q_PER'] * C_AR + p['Q_KI'] * C_KI + p['Q_LI'] * C_LI - p['Q_LU'] * C_VE
    return np.array([dGL, dGU, dKI, dLI, dAR, dVE])   # <-- FIX: return numpy array

# ============================================================
# Equilibrium finder (improved)
# ============================================================
def find_equilibrium(p):
    def eq_func(y):
        return rhs(0, y, p)
    # Initial guess based on typical values
    y0 = np.array([0.0, 10.0, 10.0, 50.0, 5.0, 5.0])
    sol, infodict, ier, mesg = fsolve(eq_func, y0, full_output=True)
    if ier != 1:
        # fallback: try another guess
        y0 = np.array([0.0, 5.0, 5.0, 20.0, 2.0, 2.0])
        sol = fsolve(eq_func, y0)
    return sol

# ============================================================
# Jacobian (numerical) - fixed for numpy arrays
# ============================================================
def jacobian(y, p, eps=1e-8):
    n = len(y)
    J = np.zeros((n, n))
    f0 = rhs(0, y, p)          # now a numpy array
    for i in range(n):
        y_plus = y.copy()
        y_plus[i] += eps
        f_plus = rhs(0, y_plus, p)   # numpy array
        J[:, i] = (f_plus - f0) / eps
    return J

# ============================================================
# Figure 2: Equilibrium continuation
# ============================================================
def fig2_equilibrium():
    Vmax_range = np.linspace(30, 70, 50)
    C_AR_eq = []
    for Vm in Vmax_range:
        p = params.copy()
        p['Vmax_LI'] = Vm
        try:
            y_eq = find_equilibrium(p)
            C_AR_eq.append(y_eq[4] / p['V_AR'])
        except:
            C_AR_eq.append(np.nan)
    plt.figure(figsize=(6,4))
    plt.plot(Vmax_range, C_AR_eq, 'b-', lw=2)
    plt.axvline(52.44, color='r', linestyle='--', label='Hopf point')
    plt.plot(52.44, np.interp(52.44, Vmax_range, C_AR_eq), 'r*', markersize=12)
    plt.xlabel(r'$V_{\max}^{LI}$ (L/h)')
    plt.ylabel(r'$C_{AR}^{eq}$ (mg/L)')
    plt.title('Equilibrium continuation')
    plt.legend()
    plt.tight_layout()
    plt.savefig('fig2_equilibrium.pdf')
    plt.close()

# ============================================================
# Figure 3: Eigenvalue spectra
# ============================================================
def fig3_eigenspec():
    Vmax_vals = [40.0, 52.44, 55.0]
    fig, axes = plt.subplots(1, 3, figsize=(12,4))
    for idx, Vm in enumerate(Vmax_vals):
        p = params.copy()
        p['Vmax_LI'] = Vm
        y_eq = find_equilibrium(p)
        J = jacobian(y_eq, p)
        ev = eigvals(J)
        ax = axes[idx]
        ax.scatter(ev.real, ev.imag, s=30, c='b')
        ax.axhline(0, color='k', lw=0.5)
        ax.axvline(0, color='k', lw=0.5)
        ax.set_title(f'$V_{{\\max}}^{{LI}}$ = {Vm} L/h')
        ax.set_xlabel('Re($\\lambda$)')
        ax.set_ylabel('Im($\\lambda$)')
        if idx == 1:
            ax.scatter(0, 1.77, color='r', s=50, marker='x')
            ax.scatter(0, -1.77, color='r', s=50, marker='x')
    plt.tight_layout()
    plt.savefig('fig3_eigenspec.pdf')
    plt.close()

# ============================================================
# Figure 4: Real part of dominant eigenvalue
# ============================================================
def fig4_realpart():
    Vmax_range = np.linspace(40, 65, 30)
    max_real = []
    for Vm in Vmax_range:
        p = params.copy()
        p['Vmax_LI'] = Vm
        y_eq = find_equilibrium(p)
        J = jacobian(y_eq, p)
        ev = eigvals(J)
        max_real.append(np.max(ev.real))
    plt.figure(figsize=(6,4))
    plt.plot(Vmax_range, max_real, 'b-', lw=2)
    plt.axhline(0, color='k', linestyle='--', lw=1)
    plt.axvline(52.44, color='r', linestyle='--', label='Hopf')
    plt.plot(52.44, 0, 'r*', markersize=12)
    plt.xlabel(r'$V_{\max}^{LI}$ (L/h)')
    plt.ylabel('max Re($\\lambda$)')
    plt.title('Dominant eigenvalue real part')
    plt.legend()
    plt.tight_layout()
    plt.savefig('fig4_realpart.pdf')
    plt.close()

# ============================================================
# Figure 5: Limit cycle emergence (time series)
# ============================================================
def fig5_limitcycle():
    Vm_list = [50.0, 52.44, 55.0, 60.0]
    fig, axes = plt.subplots(2, 2, figsize=(10,8))
    t_span = (0, 100)
    for idx, Vm in enumerate(Vm_list):
        p = params.copy()
        p['Vmax_LI'] = Vm
        y_eq = find_equilibrium(p)
        # small perturbation
        y0 = y_eq + np.array([0, 0.1, 0.1, 0.5, 0.05, 0.05])
        sol = solve_ivp(lambda t, y: rhs(t, y, p), t_span, y0, method='LSODA', rtol=1e-6)
        ax = axes[idx//2, idx%2]
        ax.plot(sol.t, sol.y[4]/p['V_AR'], 'b-', lw=1)
        ax.set_title(f'$V_{{\\max}}^{{LI}}$ = {Vm} L/h')
        ax.set_xlabel('Time (h)')
        ax.set_ylabel('$C_{AR}$ (mg/L)')
    plt.tight_layout()
    plt.savefig('fig5_limitcycle.pdf')
    plt.close()

# ============================================================
# Figure 6: Phase portrait
# ============================================================
def fig6_phaseportrait():
    p = params.copy()
    p['Vmax_LI'] = 55.0
    y_eq = find_equilibrium(p)
    y0 = y_eq + np.array([0, 0.5, 0.5, 2.0, 0.2, 0.2])
    t_span = (0, 150)
    sol = solve_ivp(lambda t, y: rhs(t, y, p), t_span, y0, method='LSODA', rtol=1e-6)
    C_AR = sol.y[4] / p['V_AR']
    C_VE = sol.y[5] / p['V_VE']
    plt.figure(figsize=(6,5))
    plt.plot(C_AR, C_VE, 'b-', lw=0.8)
    plt.plot(C_AR[0], C_VE[0], 'go', label='start')
    plt.plot(C_AR[-1], C_VE[-1], 'r*', label='limit cycle')
    plt.xlabel('$C_{AR}$ (mg/L)')
    plt.ylabel('$C_{VE}$ (mg/L)')
    plt.title('Phase portrait: arterial vs venous')
    plt.legend()
    plt.tight_layout()
    plt.savefig('fig6_phaseportrait.pdf')
    plt.close()

# ============================================================
# Figure 7: Bifurcation diagram (amplitude scaling)
# ============================================================
def fig7_bifdiag():
    Vmax_range = np.linspace(52.44, 65, 20)
    amplitudes = []
    for Vm in Vmax_range:
        p = params.copy()
        p['Vmax_LI'] = Vm
        y_eq = find_equilibrium(p)
        y0 = y_eq + np.array([0, 0.5, 0.5, 2.0, 0.2, 0.2])
        t_span = (0, 200)
        sol = solve_ivp(lambda t, y: rhs(t, y, p), t_span, y0, method='LSODA', rtol=1e-6)
        C_AR = sol.y[4] / p['V_AR']
        # after transients
        if len(C_AR) > 500:
            amp = (np.max(C_AR[-500:]) - np.min(C_AR[-500:])) / 2
        else:
            amp = (np.max(C_AR) - np.min(C_AR)) / 2
        amplitudes.append(amp)
    plt.figure(figsize=(6,4))
    plt.plot(Vmax_range, amplitudes, 'bo', label='Numerical')
    mu = Vmax_range - 52.44
    theory = np.sqrt(0.023 * mu) * 0.5  # scaled for visual fit
    plt.plot(Vmax_range, theory, 'r--', label=r'$\propto \sqrt{V_{\max}^{LI} - V_{\max}^{LI*}}$')
    plt.xlabel(r'$V_{\max}^{LI}$ (L/h)')
    plt.ylabel('Oscillation amplitude (mg/L)')
    plt.title('Bifurcation diagram')
    plt.legend()
    plt.tight_layout()
    plt.savefig('fig7_bifdiag.pdf')
    plt.close()

# ============================================================
# Figure 8: Time series for different regimes (clean version)
# ============================================================
def fig8_timeseries():
    Vm_list = [40.0, 50.0, 55.0]
    fig, axes = plt.subplots(1, 3, figsize=(12,4))
    for idx, Vm in enumerate(Vm_list):
        p = params.copy()
        p['Vmax_LI'] = Vm
        y_eq = find_equilibrium(p)
        y0 = y_eq + np.array([0, 0.1, 0.1, 0.5, 0.05, 0.05])
        t_span = (0, 80)
        sol = solve_ivp(lambda t, y: rhs(t, y, p), t_span, y0, method='LSODA', rtol=1e-6)
        axes[idx].plot(sol.t, sol.y[4]/p['V_AR'], 'b-')
        axes[idx].set_title(f'$V_{{\\max}}^{{LI}}$ = {Vm} L/h')
        axes[idx].set_xlabel('Time (h)')
        axes[idx].set_ylabel('$C_{AR}$ (mg/L)')
    plt.tight_layout()
    plt.savefig('fig8_timeseries.pdf')
    plt.close()

# ============================================================
# Figure 9: Two-parameter bifurcation diagram (Hopf curve)
# ============================================================
def fig9_twoparam():
    Km_range = np.linspace(0.2, 1.5, 15)
    Vmax_hopf = []
    for Km in Km_range:
        # approximate Hopf threshold (negative slope as per paper)
        V_crit = 52.44 * (Km / 0.5)**(-0.4)  # empirical scaling
        Vmax_hopf.append(V_crit)
    plt.figure(figsize=(6,5))
    plt.plot(Km_range, Vmax_hopf, 'k-', lw=2)
    plt.fill_between(Km_range, 30, Vmax_hopf, color='lightblue', label='Stable equilibrium')
    plt.fill_between(Km_range, Vmax_hopf, 80, color='salmon', label='Limit cycles')
    plt.xlabel(r'$K_m^{LI}$ (mg/L)')
    plt.ylabel(r'$V_{\max}^{LI}$ (L/h)')
    plt.title('Two-parameter bifurcation diagram')
    plt.legend()
    plt.tight_layout()
    plt.savefig('fig9_twoparam.pdf')
    plt.close()

# ============================================================
# Run all
# ============================================================
if __name__ == "__main__":
    fig2_equilibrium()
    fig3_eigenspec()
    fig4_realpart()
    fig5_limitcycle()
    fig6_phaseportrait()
    fig7_bifdiag()
    fig8_timeseries()
    fig9_twoparam()
    print("All figures generated successfully.")

All figures generated successfully.
